# TinyImageNet Canonical ALiBi — Parallel Runner (1D OR 2D)

Notebook for running **one** of the two paired TinyImageNet ALiBi
trainings — either 1D-ALiBi or 2D-ALiBi — in a single Colab session.
Use **two copies of this notebook** in parallel (one configured as 1D,
the other as 2D) to cut wall time roughly in half: ~22 h instead of
~44 h sequentially.

**Both sessions write into the SAME results directory**
(`Trained models_TinyImageNet_v2/`), into disjoint subfolders
(`alibi_seed42/` and `alibi_2d_seed42/`), so the results merge
naturally and can be analysed together when both finish.

## Setup before running both sessions in parallel
1. Run **cells 1–4** (mount, GPU check, stage scripts + apply patch, **dataset prep**) in **one** session first and let cell 4 finish before launching the second session. This avoids the download/extract race condition on Google Drive.
2. Once cell 4 has completed, open this notebook as a **second** session (different runtime / different account), repeat cells 1–3 (cell 4 will be a no-op because the dataset is already prepared), and in cell 5 set `PE_VARIANT` to the **other** value than what the first session is running.
3. Run cell 5 in each session to create the per-variant script, then cell 6 to launch training.

## Colab plan caveat
Free / Pro Colab usually does **not** allow two simultaneous GPU sessions on the same account — the second one will kick the first off. Run the two sessions either on Pro+ (sometimes allows two), or on two separate accounts, or sequentially on one account if simultaneous is blocked.

## 1. Mount Drive + GPU check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'No GPU!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')


## 2. Install deps + stage scripts + (re)generate patched module

Identical to the sequential notebook. Safe to run in both sessions:
Drive reads are concurrent-safe, and the patched module is written into
`/content/` (per-session storage, not shared between sessions).

In [ ]:
!pip install -q timm tqdm

import os, shutil

DRIVE_BASE = '/content/drive/MyDrive'
PE_EXPERIMENT_BASE = os.path.join(DRIVE_BASE, 'pe_experiment')

needed = [
    ('apply_2d_alibi_patch.py', [
        os.path.join(DRIVE_BASE, 'apply_2d_alibi_patch.py'),
        os.path.join(PE_EXPERIMENT_BASE, 'apply_2d_alibi_patch.py'),
    ]),
    ('full_scale_experiment.py', [
        os.path.join(PE_EXPERIMENT_BASE, 'full_scale_experiment.py'),
        os.path.join(DRIVE_BASE, 'full_scale_experiment.py'),
    ]),
    ('tinyimagenet_alibi_canonical.py', [
        os.path.join(DRIVE_BASE, 'tinyimagenet_alibi_canonical.py'),
        os.path.join(PE_EXPERIMENT_BASE, 'tinyimagenet_alibi_canonical.py'),
    ]),
]

for fname, candidates in needed:
    src = next((p for p in candidates if os.path.isfile(p)), None)
    if src is None:
        raise FileNotFoundError(f'Cannot find {fname} in any of: {candidates}')
    shutil.copy(src, f'/content/{fname}')
    print(f'  Copied: {src} -> /content/{fname}')

exit_code = os.system(
    'cd /content && python apply_2d_alibi_patch.py '
    'full_scale_experiment.py full_scale_experiment_v2.py')
if exit_code != 0:
    raise RuntimeError(f'Patch failed with exit code {exit_code}')

assert os.path.isfile('/content/full_scale_experiment_v2.py')
print('\nReady.')


## 3. Pick variant (1D or 2D)

**Set `PE_VARIANT` to either `'1d'` or `'2d'`.** This is the ONLY
difference between the two parallel notebooks.

It controls:
- which `PE_TYPES` value the variant script will train
- the output sub-folder under `Trained models_TinyImageNet_v2/`


In [ ]:
PE_VARIANT = '1d'   # <<<<<<<<<< CHANGE TO '2d' IN THE OTHER NOTEBOOK

assert PE_VARIANT in ('1d', '2d'), "PE_VARIANT must be '1d' or '2d'"
print(f"This session will train: {'1D-ALiBi' if PE_VARIANT == '1d' else '2D-ALiBi'}")


## 4. Prepare TinyImageNet dataset (run once, then safe to skip)

**Critical:** run this in **one** session and let it finish (~2 min if
downloading, instantly if already prepared) BEFORE launching the second
session's cell 6. After that, both sessions can safely run cell 4 —
the second one becomes a no-op because `val_reorganised/` already exists.

Doing this avoids the download/extract race condition where two
sessions try to write into the same `tiny-imagenet-200.zip` on Drive.

In [ ]:
import sys
sys.path.insert(0, '/content')
from tinyimagenet_alibi_canonical import download_and_prepare, DATA_DIR

extracted = download_and_prepare(DATA_DIR)
print(f'\nTinyImageNet prepared at: {extracted}')


## 5. Create the per-variant training script

Makes a small copy of `tinyimagenet_alibi_canonical.py` with `PE_TYPES`
narrowed to just the chosen variant. The original master script is NOT
modified — it remains usable for the sequential 2-PE workflow.

The copy lives only in `/content/` (per-session storage), so each
parallel session has its own. Both copies write training outputs into
the same Drive folder (`Trained models_TinyImageNet_v2/`) but into
disjoint sub-folders, so they don't conflict.

In [ ]:
src = '/content/tinyimagenet_alibi_canonical.py'
dst = f'/content/tin_run_{PE_VARIANT}.py'

with open(src) as f:
    text = f.read()

ORIGINAL = "PE_TYPES = ['alibi', 'alibi_2d']"
if PE_VARIANT == '1d':
    REPLACEMENT = "PE_TYPES = ['alibi']"
else:
    REPLACEMENT = "PE_TYPES = ['alibi_2d']"

if ORIGINAL not in text:
    raise RuntimeError(
        f'Could not find expected line in master script: {ORIGINAL!r}.\n'
        f'Has the master been edited? Check tinyimagenet_alibi_canonical.py.'
    )

patched = text.replace(ORIGINAL, REPLACEMENT, 1)
with open(dst, 'w') as f:
    f.write(patched)

# Sanity-check that the replacement landed and nothing else changed
assert REPLACEMENT in patched, 'replacement not present in output'
assert patched.count('PE_TYPES =') == 1, \
    'multiple PE_TYPES assignments after patching -- aborting'

print(f'Created {dst} with PE_TYPES = {REPLACEMENT.split("=")[1].strip()}')
print(f'(diff from master: {len(patched) - len(text):+d} chars)')


## 6. Run training (this session's variant only)

Streams epoch-by-epoch output. Has resume logic via the canonical
script: if you re-launch this cell after a Colab disconnect, completed
epochs are restored and training continues from the last full epoch in
`training_history.json`.

Expected wall time: ~19 h on G4 (RTX 6000 Blackwell).

In [ ]:
import subprocess, sys

script_path = f'/content/tin_run_{PE_VARIANT}.py'
cmd = f'cd /content && python -u {script_path}'

print('=' * 70)
print(f'TinyImageNet canonical training -- variant: {PE_VARIANT.upper()}')
print(f'Script: {script_path}')
print('=' * 70)

p = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
)
for line in p.stdout:
    print(line, end='', flush=True)
    sys.stdout.flush()
exit_code = p.wait()

print('\n' + '=' * 70)
print('ALL DONE' if exit_code == 0 else f'Exited with code {exit_code}')
print('=' * 70)


## 7. Status check (cross-session view)

Shows status of BOTH variants (this session's and the parallel session's),
since both write to the same Drive folder. Safe to run any time, in either
notebook. Reports the paired 2D-vs-1D delta once both runs finish.

In [ ]:
import os, json

V2_RESULTS  = '/content/drive/MyDrive/Trained models_TinyImageNet_v2'
OLD_RESULTS = '/content/drive/MyDrive/Trained models_TinyImageNet'
SEED = 42

def best_acc(results_dir, pe, seed, min_epochs=300):
    p = os.path.join(results_dir, f'{pe}_seed{seed}', 'training_history.json')
    if not os.path.isfile(p):
        return None, 0
    with open(p) as f:
        h = json.load(f)
    accs = h.get('val_acc', [])
    return (max(accs) if accs else None), len(accs)

print('TinyImageNet canonical ALiBi training status (both variants):')
print('=' * 70)
a1, n1 = best_acc(V2_RESULTS, 'alibi',    SEED)
a2, n2 = best_acc(V2_RESULTS, 'alibi_2d', SEED)

def fmt(a, n):
    if a is None:
        return '   -- (not started)'
    status = 'DONE' if n >= 300 else f'partial {n}/300'
    return f'{a:6.2f}%  [{status}]'

print(f'  1D-ALiBi (canonical, seed {SEED}): {fmt(a1, n1)}')
print(f'  2D-ALiBi (canonical, seed {SEED}): {fmt(a2, n2)}')

if a1 is not None and a2 is not None and n1 >= 300 and n2 >= 300:
    d = a2 - a1
    print(f'\n  Paired delta (2D - 1D): {d:+.2f} pp')
elif n1 + n2 > 0:
    print(f'\n  Both runs not yet complete; paired delta not computed.')

# Sanity check vs. legacy (original protocol) 1D-ALiBi from the old
# results dir, to see how much grad-clip + no-compile shifts the
# 1D-ALiBi number relative to the existing Table 3 entry (53.41%).
print('\n' + '-' * 70)
print('Sanity check vs. legacy (original protocol) 1D-ALiBi TIN result:')
a_legacy, n_legacy = best_acc(OLD_RESULTS, 'alibi', SEED)
if a1 is not None and a_legacy is not None:
    delta = a1 - a_legacy
    print(f'  seed {SEED}: canonical={a1:.2f}%  legacy={a_legacy:.2f}%  '
          f'diff={delta:+.2f} pp')
    if abs(delta) > 2.0:
        print(f'  [!] Difference > 2 pp -- worth investigating which training '
              f'detail (grad-clip vs. torch.compile) caused it.')
elif a_legacy is not None:
    print(f'  seed {SEED}: legacy={a_legacy:.2f}%, canonical not yet done')
else:
    print(f'  seed {SEED}: no comparison possible yet (neither result present)')
